# 챔피언 스코어 v2 — Hybrid (팀 보정 + 블렌드 가중치)

내전 데이터 기준 **라인별 챔피언 성능 순위**. v1 대비 개선점:

| 영역 | v1 | **v2 (이 노트북)** |
|---|---|---|
| 지표 가중치 | 손으로 잡은 직관값 | **데이터(승패예측) + 전문가값 블렌드** |
| 스케일 | 평균/표준편차 z-score | **로버스트(median/IQR) + 클리핑** |
| V1 승률 | 플레이어 평균 대비 | **팀 실력 보정** (우리팀·상대팀 기대승률 대비) |
| V1:V2 비중 | 50:50 | **35:65** (내전은 팀빨 노이즈↑ → 개인 퍼포먼스 신뢰) |
| 지표 구성 | vision 과대·dpm 과대 | vision 강등, dead_time_pct 추가, dpm↓ exp/min·lane_gold_diff↑ |
| STABLE_GAMES | 임의(30) | **데이터 중앙값 기반** |

## 왜 바꿨나 — 데이터가 말한 것
각 지표 vs 승패 상관을 보니 통념과 달랐다:
- `vision_score`: 승패 상관 **0.06~0.10 (거의 무의미)** → 서포터 앵커에서 강등
- `dpm`: **0.12~0.37 (약함)** → 딜량은 생각보다 승패를 못 가름 → 하향
- `exp_per_min`(0.54~0.68)·`lane_gold_diff`(0.47~0.70)가 **최강** → 상향, 특히 SUP의 lane_gold_diff
- `dead_time_pct`(-0.42~-0.54): 강한 신호였는데 누락 → 추가

> **순환성 주의**: kda·exp/min·gold/min 등은 *이겨서 오르는* 지표이기도 하다.
> 그래서 "승패 상관 = 가중치"로 직결하지 않고 데이터값과 전문가값을 **블렌드**해
> V2가 V1(승률)을 그대로 재복사하지 않도록 했다.


# 0. 환경설정

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# 1. 데이터 로드 & 점검

In [2]:
PATH = "all_participant_metric_data.csv"   # 경로만 본인 환경에 맞게
df = pd.read_csv(PATH)
df = df[df["is_deleted"] == False].copy()

print("shape:", df.shape)
print("경기 수:", df["custom_match_id"].nunique(), "| 팀당 5명 x 2팀 구조")
print("포지션 :", df["position"].value_counts().to_dict())

shape: (37030, 79)
경기 수: 3703 | 팀당 5명 x 2팀 구조
포지션 : {'TOP': 7406, 'JUG': 7406, 'MID': 7406, 'ADC': 7406, 'SUP': 7406}


# 2. 파라미터 · 지표 방향 · 전문가 가중치

튜닝값은 전부 여기서 관리한다.

In [3]:
# ---- 공통 파라미터 ----
CHAMP_KEY, CHAMP_NAME = "champion_id", "champ_name"
MIN_GAMES  = 10            # 최소 표본 (오프메타 소표본을 더 거르려면 20~30)
W_V1, W_V2 = 0.35, 0.65    # 하이브리드 비중 (내전=팀빨↑ 이라 V2 우위)
ALPHA      = 0.5           # 가중치 블렌드: 데이터 vs 전문가 (1=데이터만, 0=전문가만)
RZ_CLIP    = 4             # 로버스트 스케일 클리핑 한계 (폭주판 방지)
POSITION_ORDER = ["TOP", "JUG", "MID", "ADC", "SUP"]

# ---- 지표와 방향 (-1 = 낮을수록 좋음) ----
METRIC_DIR = {
    "dpm": 1, "kda": 1, "damage_dealt_per_death": 1,
    "gold_per_min": 1, "cs_per_min": 1, "exp_per_min": 1,
    "lane_gold_diff": 1, "damage_to_objectives": 1, "takedowns_before_15min": 1,
    "dead_time_pct": -1,           # 죽어있는 시간 = 적을수록 좋음
    "vision_score": 1,             # 예측력 낮음(상징적 소량 반영)
}
METRICS = list(METRIC_DIR)

# ---- 수정 전문가 가중치 (0~1) : 데이터 교훈 반영 + 역할 철학 ----
EXPERT = {
 "TOP":{"dpm":0.55,"kda":0.55,"damage_dealt_per_death":0.75,"gold_per_min":0.70,"cs_per_min":0.55,"exp_per_min":0.85,"lane_gold_diff":0.95,"damage_to_objectives":0.60,"takedowns_before_15min":0.55,"dead_time_pct":0.70,"vision_score":0.25},
 "JUG":{"dpm":0.45,"kda":0.60,"damage_dealt_per_death":0.60,"gold_per_min":0.65,"cs_per_min":0.45,"exp_per_min":0.85,"lane_gold_diff":0.80,"damage_to_objectives":0.95,"takedowns_before_15min":0.85,"dead_time_pct":0.70,"vision_score":0.40},
 "MID":{"dpm":0.70,"kda":0.55,"damage_dealt_per_death":0.70,"gold_per_min":0.70,"cs_per_min":0.45,"exp_per_min":0.85,"lane_gold_diff":0.90,"damage_to_objectives":0.60,"takedowns_before_15min":0.65,"dead_time_pct":0.70,"vision_score":0.30},
 "ADC":{"dpm":0.75,"kda":0.55,"damage_dealt_per_death":0.75,"gold_per_min":0.85,"cs_per_min":0.65,"exp_per_min":0.85,"lane_gold_diff":0.90,"damage_to_objectives":0.80,"takedowns_before_15min":0.50,"dead_time_pct":0.70,"vision_score":0.25},
 "SUP":{"dpm":0.30,"kda":0.60,"damage_dealt_per_death":0.50,"gold_per_min":0.45,"cs_per_min":0.15,"exp_per_min":0.80,"lane_gold_diff":0.90,"damage_to_objectives":0.55,"takedowns_before_15min":0.55,"dead_time_pct":0.70,"vision_score":0.45},
}

# 3. 로버스트 스케일 (포지션 내)

지표 대부분이 오른쪽으로 심하게 치우쳐 있어(damage_dealt_per_death 왜도 3.0, kda 2.5)
평균/표준편차 z-score는 소수 폭주판이 지배한다. **median/IQR** 로버스트 스케일 + 클리핑으로 완화.

In [4]:
# 방향 정렬: 모두 '높을수록 좋음'
for m, d in METRIC_DIR.items():
    if d == -1:
        df[m] = -df[m]

# 포지션 내 로버스트 스케일 (median/IQR)
def robust(s):
    iqr = s.quantile(0.75) - s.quantile(0.25)
    return (s - s.median()) / iqr if iqr > 0 else s * 0.0

for m in METRICS:
    df[f"{m}_rz"] = df.groupby("position")[m].transform(robust).clip(-RZ_CLIP, RZ_CLIP)

# 4. 데이터 기반 가중치 → 전문가값과 블렌드

포지션별 **릿지 로지스틱 회귀**(game_result ~ 스케일된 지표)의 계수를 데이터 가중치로 쓴다.
음수(다른 지표 통제 시 역효과) 계수는 0으로 클립, 라인별 최댓값=1로 정규화.
전문가값도 라인별 최댓값=1로 정규화 후 ALPHA 로 블렌드.

In [5]:
data_w = {}
for pos in POSITION_ORDER:
    d = df[df["position"] == pos]
    X = d[[f"{m}_rz" for m in METRICS]].fillna(0).values
    y = d["game_result"].values
    lr = LogisticRegression(C=0.5, max_iter=1000).fit(X, y)
    coef = np.clip(lr.coef_[0], 0, None)
    data_w[pos] = dict(zip(METRICS, coef / coef.max() if coef.max() > 0 else coef))

ROLE_W = {}
for pos in POSITION_ORDER:
    ew = EXPERT[pos]; mx = max(ew.values())
    ROLE_W[pos] = {m: ALPHA*data_w[pos][m] + (1-ALPHA)*(ew[m]/mx) for m in METRICS}

print("=== 최종 블렌드 가중치 (라인별) ===")
pd.DataFrame(ROLE_W).round(2)

=== 최종 블렌드 가중치 (라인별) ===


,TOP,JUG,MID,ADC,SUP
dpm,0.29,0.24,0.39,0.45,0.17
kda,0.79,0.82,0.81,0.81,0.73
damage_dealt_per_death,0.39,0.32,0.39,0.42,0.28
gold_per_min,0.37,0.34,0.39,0.47,0.36
cs_per_min,0.29,0.24,0.25,0.36,0.08
exp_per_min,0.91,0.87,0.69,0.58,0.62
lane_gold_diff,0.73,0.59,0.62,0.74,1.00
damage_to_objectives,0.44,0.65,0.42,0.63,0.37
takedowns_before_15min,0.29,0.45,0.36,0.28,0.31
dead_time_pct,0.40,0.39,0.40,0.44,0.48


# 5. V2 — 퍼포먼스 점수
포지션별 블렌드 가중치로 스케일된 지표를 가중합.

In [6]:
w_arr = {pos: np.array([ROLE_W[pos][m] for m in METRICS]) for pos in POSITION_ORDER}
rz = df[[f"{m}_rz" for m in METRICS]].values
df["perf"] = [float(v @ w_arr[p]) for v, p in zip(rz, df["position"])]

# 6. V1 — 팀 실력 보정 승률

내전은 고정/사전구성 팀이 많아 승패가 팀 실력에 크게 좌우된다. 각 경기에서
**우리 팀 5인 평균 기대승률 - 상대 팀 5인 평균**으로 기대 결과를 만들고,
실제 결과에서 뺀 잔차(surprise)를 챔피언의 승률 기여로 본다.

expected = 0.5 + (우리5평균 - 상대5평균),  surprise = 결과 - expected

In [7]:
pw = df.groupby("puuid")["game_result"].mean()
df["p"] = df["puuid"].map(pw)

tsum = df.groupby(["custom_match_id", "game_team"])["p"].transform("sum")
match_tot = df.groupby("custom_match_id")["p"].transform("sum")
df["team_mean"]  = tsum / 5.0
df["enemy_mean"] = (match_tot - tsum) / 5.0
df["expected"]   = (0.5 + (df["team_mean"] - df["enemy_mean"])).clip(0.02, 0.98)
df["surprise"]   = df["game_result"] - df["expected"]

# 7. 하이브리드 결합 & 0~100 점수화

챔피언-포지션 집계 -> V1·V2 각각 라인 내 표준화 -> 0.35·z(V1)+0.65·z(V2)
-> 표본 신뢰도 min(games/STABLE_GAMES,1) 로 소표본 중앙 수축 -> 라인 내 70±15 변환.

STABLE_GAMES는 데이터의 게임 수 중앙값으로 자동 설정.

In [8]:
champ = df.groupby([CHAMP_KEY, "position"], as_index=False).agg(
    champ_name=(CHAMP_NAME, "first"),
    games=("game_result", "count"),
    unique_users=("puuid", "nunique"),
    champion_winrate=("game_result", "mean"),
    v1_core=("surprise", "mean"),
    v2_core=("perf", "mean"),
)
champ = champ[champ["games"] >= MIN_GAMES].copy()

STABLE_GAMES = int(champ["games"].median())
print("STABLE_GAMES (게임수 중앙값):", STABLE_GAMES)

def z_pos(s):
    sd = s.std(ddof=0)
    return (s - s.mean()) / sd if sd > 0 else s * 0.0

champ["z1"] = champ.groupby("position")["v1_core"].transform(z_pos)
champ["z2"] = champ.groupby("position")["v2_core"].transform(z_pos)
champ["reliability"] = np.minimum(champ["games"] / STABLE_GAMES, 1.0)
champ["blend"] = W_V1 * champ["z1"] + W_V2 * champ["z2"]
champ["final"] = champ["blend"] * champ["reliability"]
champ["champion_score"] = (
    champ.groupby("position")["final"]
         .transform(lambda s: 70 + z_pos(s) * 15).clip(0, 100).round(2)
)

for c in ["champion_winrate", "v1_core", "v2_core"]:
    champ[c] = champ[c].round(3)

final_score = champ.sort_values(["position", "champion_score"],
                                ascending=[True, False]).reset_index(drop=True)
DISPLAY = ["champ_name", CHAMP_KEY, "position", "games", "unique_users",
           "champion_winrate", "v1_core", "v2_core", "champion_score"]
print("총 챔피언-포지션:", len(final_score))
final_score["champion_score"].describe().round(2)

STABLE_GAMES (게임수 중앙값): 80
총 챔피언-포지션: 258


count    258.00
mean      69.66
std       14.07
min       21.52
25%       60.55
50%       69.01
75%       77.80
max      100.00
Name: champion_score, dtype: float64

# 8. 라인별 결과 & 저장

## 8.1 TOP

In [9]:
result_top = final_score[final_score["position"]=="TOP"][DISPLAY].reset_index(drop=True)
result_top

,champ_name,champion_id,position,games,unique_users,champion_winrate,v1_core,v2_core,champion_score
0,가렌,CHN_38,TOP,82,29,0.598,0.094,2.323,100.00
1,갱플랭크,CHN_42,TOP,99,32,0.566,0.060,2.128,100.00
2,올라프,CHN_97,TOP,147,42,0.544,0.036,2.118,100.00
3,이렐리아,CHN_48,TOP,95,34,0.537,0.038,1.905,97.76
4,카밀,CHN_22,TOP,167,43,0.575,0.079,1.569,96.40
...,...,...,...,...,...,...,...,...,...
60,말파이트,CHN_78,TOP,345,133,0.478,-0.014,-0.023,52.21
61,럼블,CHN_111,TOP,284,76,0.437,-0.061,0.243,51.25
62,그라가스,CHN_40,TOP,96,53,0.417,-0.077,-0.085,42.48
63,세트,CHN_119,TOP,82,36,0.390,-0.104,-0.278,35.11


## 8.2 JUNGLE

In [10]:
result_jug = final_score[final_score["position"]=="JUG"][DISPLAY].reset_index(drop=True)
result_jug

,champ_name,champion_id,position,games,unique_users,champion_winrate,v1_core,v2_core,champion_score
0,제드,CHN_165,JUG,112,45,0.580,0.080,2.152,100.00
1,릴리아,CHN_73,JUG,115,57,0.548,0.047,1.702,97.17
2,자헨,CHN_172,JUG,190,54,0.563,0.070,1.530,96.56
3,그레이브즈,CHN_41,JUG,269,75,0.491,-0.007,2.003,96.42
4,제이스,CHN_53,JUG,209,80,0.502,-0.009,1.865,93.23
5,탈리야,CHN_134,JUG,47,26,0.553,0.040,2.017,89.17
6,니달리,CHN_93,JUG,56,25,0.518,0.026,1.708,87.11
7,카직스,CHN_65,JUG,226,59,0.509,0.006,1.445,86.49
8,잭스,CHN_52,JUG,49,24,0.592,0.091,1.383,85.99
9,자이라,CHN_170,JUG,35,16,0.657,0.156,1.396,85.16


## 8.3 MIDDLE

In [11]:
result_mid = final_score[final_score["position"]=="MID"][DISPLAY].reset_index(drop=True)
result_mid

,champ_name,champion_id,position,games,unique_users,champion_winrate,v1_core,v2_core,champion_score
0,카사딘,CHN_60,MID,91,35,0.692,0.177,2.593,100.00
1,제라스,CHN_158,MID,221,80,0.566,0.066,1.453,97.62
2,르블랑,CHN_70,MID,234,73,0.538,0.038,1.300,90.85
3,제드,CHN_165,MID,134,46,0.552,0.052,1.217,90.35
4,트리스타나,CHN_139,MID,37,24,0.568,0.079,2.069,89.60
...,...,...,...,...,...,...,...,...,...
57,애니,CHN_7,MID,130,47,0.500,-0.006,-0.126,51.89
58,사일러스,CHN_131,MID,299,107,0.485,-0.012,-0.126,51.28
59,아지르,CHN_14,MID,137,46,0.350,-0.142,0.021,40.38
60,갈리오,CHN_37,MID,406,120,0.473,-0.031,-0.527,39.61


## 8.4 ADC

In [12]:
result_adc = final_score[final_score["position"]=="ADC"][DISPLAY].reset_index(drop=True)
result_adc

,champ_name,champion_id,position,games,unique_users,champion_winrate,v1_core,v2_core,champion_score
0,제리,CHN_166,ADC,135,49,0.541,0.033,1.467,100.00
1,자야,CHN_157,ADC,180,76,0.533,0.037,1.216,97.21
2,스몰더,CHN_127,ADC,285,94,0.568,0.069,0.839,92.60
3,카이사,CHN_56,ADC,303,105,0.495,-0.010,1.378,90.82
4,스웨인,CHN_130,ADC,53,20,0.623,0.111,0.817,89.73
5,유나라,CHN_171,ADC,397,90,0.511,0.011,1.121,87.39
6,코그모,CHN_68,ADC,59,17,0.542,0.035,1.054,84.15
7,코르키,CHN_25,ADC,111,39,0.505,-0.004,1.071,81.88
8,루시안,CHN_75,ADC,419,124,0.499,-0.001,0.976,79.26
9,세나,CHN_117,ADC,105,48,0.514,0.004,0.879,77.31


## 8.5 SUPPORT

In [13]:
result_sup = final_score[final_score["position"]=="SUP"][DISPLAY].reset_index(drop=True)
result_sup

,champ_name,champion_id,position,games,unique_users,champion_winrate,v1_core,v2_core,champion_score
0,자이라,CHN_170,SUP,87,26,0.575,0.070,2.174,100.00
1,이즈리얼,CHN_33,SUP,35,15,0.600,0.093,4.550,100.00
2,유미,CHN_163,SUP,248,59,0.560,0.056,1.515,90.47
3,벨코즈,CHN_149,SUP,30,7,0.600,0.103,2.419,88.69
4,제라스,CHN_158,SUP,26,12,0.615,0.084,2.891,88.65
5,세나,CHN_117,SUP,55,34,0.473,-0.021,2.139,85.27
6,엘리스,CHN_31,SUP,63,32,0.540,0.032,1.521,83.85
7,파이크,CHN_102,SUP,110,53,0.555,0.053,1.153,82.75
8,소나,CHN_128,SUP,111,36,0.568,0.064,1.069,82.74
9,모르가나,CHN_83,SUP,112,48,0.509,0.003,1.374,79.51


## 8.6 CSV 저장

In [14]:
for pos in POSITION_ORDER:
    (final_score[final_score["position"]==pos][DISPLAY]
     .to_csv(f"champion_score_{pos.lower()}_v2.csv", index=False, encoding="utf-8-sig"))
final_score[DISPLAY].to_csv("champion_score_all_v2.csv", index=False, encoding="utf-8-sig")
print("저장 완료: champion_score_{top,jug,mid,adc,sup,all}_v2.csv")

저장 완료: champion_score_{top,jug,mid,adc,sup,all}_v2.csv


# 9. 한계 & 튜닝 노트

- **champion_score(0~100)는 "라인 내 상대 순위"** 지 절대 실력치가 아니다. 라인 안 챔프들이
  다 비슷해도 30~100으로 벌어진다. 웹 게시 시 "라인 내 상대 평가"로 표기 권장.
- **kda의 순환성**: 이기면 자연히 좋아지는 지표라 데이터 가중치에서 높게 잡힌다.
  V1(승률)과 겹치므로 더 줄이려면 ALPHA를 낮추거나 METRIC_DIR에서 kda를 빼고 재적합.
- **오프메타 소표본**(예: 서폿 이즈리얼): 역할 정규화 지표가 커져 상위로 튄다.
  걸러내려면 MIN_GAMES를 20~30으로.
- **주요 튜닝 레버**: W_V1/W_V2(승률 vs 퍼포먼스), ALPHA(데이터 vs 전문가),
  MIN_GAMES(소표본 필터), EXPERT(라인별 철학), RZ_CLIP(폭주 억제).
